# Entrenar la voz propia de Asia (Piper) en Colab

Para Colab gratuito con GPU T4: **Entorno de ejecución → Cambiar tipo → T4 GPU**, y luego ejecuta las celdas **en orden**.
Fine-tuning de Piper (piper1-gpl) desde un checkpoint medium en español, con los checkpoints en tu Drive para poder
**reanudar** en otra sesión (Colab corta hacia las 4 h): vuelve a ejecutar todo y sigue desde el último.

Antes de empezar:
1. Sube a Drive, en `MyDrive/asia_voz/entrada/`, el zip del dataset (`fa_normal_22050.zip` o `fa_atenuado_22050.zip`, de `dataset_final.py`).
2. La voz es de otra persona, con permiso y para uso **privado**. El zip solo debe subirse a tu Drive con su visto bueno.
3. La última celda borra los audios de Drive y de Colab. No ejecutes el cuaderno compartido ni lo publiques con salidas.

Pasos y tiempos estimados: `docs/VOZ_PROPIA.md` en el repositorio de Asia.

## 0. Configuración

In [ ]:
# Lo que se cambia de una sesión a otra.
DATASET_ZIP = "fa_normal_22050.zip"   # o "fa_atenuado_22050.zip" (silencios atenuados)
VOICE_NAME = "es_MX-propia"
BASE = "ald"                          # "ald" (es_MX, hombre; recomendado) o "sharvard" (plan B: es_ES, 2 hablantes)
ESPEAK_VOICE = "es-419"               # español de América Latina
BATCH_SIZE = 16
MAX_EPOCHS = 1000                     # TOTAL, contando las de sesiones anteriores (un checkpoint reanudado cuenta las suyas)
SESSION_MAX_TIME = "00:03:30:00"      # DD:HH:MM:SS; para antes del corte de Colab (~4 h) y deja un último checkpoint
SYNC_EVERY_S = 300                    # cada cuánto se copia last.ckpt a Drive

DRIVE_DIR = "/content/drive/MyDrive/asia_voz"
LOCAL = "/content"

# Checkpoints de base (rhasspy/piper-checkpoints, es). Se bajan a Colab, no a tu PC.
HF = "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/es"
BASES = {
    "ald": {
        "path": "es_MX/ald/medium", "ckpt": "epoch%3D9999-step%3D1753600.ckpt",
        "sha256": "77ee30f9a382594123ec4fb3ca53b437612819a44b3e3dc490f39f5a2dfefcce",
    },
    "sharvard": {
        "path": "es_ES/sharvard/medium", "ckpt": "epoch%3D4899-step%3D215600.ckpt",
        "sha256": "cae76056c432f1a477705af1ebec55b73b586865e73b256d4bdc6fdaea8296c9",
    },
}
assert BASE in BASES, BASE

## 1. GPU y Drive

In [ ]:
import subprocess, sys
print(sys.version)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

for folder in ("entrada", "base", "checkpoints", "salida"):
    os.makedirs(f"{DRIVE_DIR}/{folder}", exist_ok=True)

print(os.listdir(f"{DRIVE_DIR}/entrada"))

## 2. Instalación (solo en Colab)

In [ ]:
!apt-get -qq install -y build-essential cmake ninja-build espeak-ng aria2 > /dev/null
%cd /content
!git clone -q https://github.com/OHF-voice/piper1-gpl.git
%cd /content/piper1-gpl
!python3 -m pip install -q -e '.[train]'
!./build_monotonic_align.sh
!python3 setup.py build_ext --inplace

In [ ]:
# Con PyTorch 2.6 o posterior, cargar un checkpoint falla por torch.load(weights_only=True)
# (discusión #138 de piper1-gpl). Esta variable de PyTorch lo desactiva sin tocar el código de Piper.
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

import torch
print("torch", torch.__version__, "· GPU:", torch.cuda.is_available())

## 3. Dataset y checkpoint de base

In [ ]:
import glob, shutil, zipfile

zip_path = f"{DRIVE_DIR}/entrada/{DATASET_ZIP}"
assert os.path.exists(zip_path), f"Falta {zip_path}: súbelo a Drive (carpeta asia_voz/entrada)"

shutil.rmtree(f"{LOCAL}/dataset", ignore_errors=True)

with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(f"{LOCAL}/dataset")

lines = open(f"{LOCAL}/dataset/metadata.csv", encoding="utf-8").read().splitlines()
wavs = glob.glob(f"{LOCAL}/dataset/wavs/*.wav")
print(len(lines), "frases;", len(wavs), "wavs")
assert len(lines) == len(wavs) > 0

In [ ]:
import hashlib

info = BASES[BASE]
base_dir = f"{DRIVE_DIR}/base/{BASE}"
os.makedirs(base_dir, exist_ok=True)
base_ckpt = f"{base_dir}/base.ckpt"

# aria2c y no wget: wget corrompe las descargas de Hugging Face en Colab (el archivo empieza por "<" en vez de "PK").
if not os.path.exists(base_ckpt):
    !aria2c -x 8 -s 8 -q -d {base_dir} -o base.ckpt "{HF}/{info['path']}/{info['ckpt']}"

with open(base_ckpt, "rb") as f:
    assert f.read(2) == b"PK", "La descarga no es un checkpoint (¿página de error?)"

digest = hashlib.sha256()

with open(base_ckpt, "rb") as f:
    for block in iter(lambda: f.read(1 << 24), b""):
        digest.update(block)

assert digest.hexdigest() == info["sha256"], "sha256 distinto: descarga corrupta, bórrala y repite"
print("base:", BASE, f"{os.path.getsize(base_ckpt) / 1e6:.0f} MB, sha256 correcto")

## 4. Entrenamiento (reanudable)

- Primera vez: `--model.warmstart_ckpt` copia solo los pesos de la base y empieza en la época 0 (los checkpoints viejos de
  `piper-checkpoints` no se pueden reanudar con `--ckpt_path`, solo usar de base).
- Si ya hay un `last.ckpt` en Drive: `--ckpt_path` reanuda la época, el optimizador y el scheduler. `MAX_EPOCHS` debe incluir las ya hechas.
- Piper reescribe `last.ckpt` en cada época. Entrena en disco local y copia `last.ckpt` a Drive cada `SYNC_EVERY_S` s
  (copia a un archivo temporal y renombra, para no dejar un checkpoint a medias si Colab se corta).
- Escucha a las 300, 500 y 1000 épocas (sección 5): con tan poco audio suele bastar antes de 2000.
- Plan B (`BASE = "sharvard"`, dos hablantes): cargar sus pesos en un modelo de un hablante puede fallar por la forma de las capas;
  no está probado. Si falla, revisa `docs/VOZ_PROPIA.md`.

In [ ]:
import subprocess, time, glob

CKPT_DRIVE = f"{DRIVE_DIR}/checkpoints/last.ckpt"
LOGS = f"{LOCAL}/entrenamiento"
CONFIG = f"{DRIVE_DIR}/salida/{VOICE_NAME}-medium.onnx.json"


def newest_last():
    found = glob.glob(f"{LOGS}/lightning_logs/version_*/checkpoints/last.ckpt")
    return max(found, key=os.path.getmtime) if found else None


def sync():
    """last.ckpt local -> Drive, de forma atómica."""
    source = newest_last()

    if source:
        shutil.copyfile(source, CKPT_DRIVE + ".tmp")
        os.replace(CKPT_DRIVE + ".tmp", CKPT_DRIVE)
        print(f"[sync] {os.path.basename(os.path.dirname(os.path.dirname(source)))} -> Drive", time.strftime("%H:%M:%S"))


if os.path.exists(CKPT_DRIVE):
    start = ["--ckpt_path", CKPT_DRIVE]
    print("Reanudando desde", CKPT_DRIVE)
else:
    start = ["--model.warmstart_ckpt", base_ckpt]
    print("Empezando desde la base", BASE)

command = [
    sys.executable, "-m", "piper.train", "fit",
    "--data.voice_name", VOICE_NAME,
    "--data.csv_path", f"{LOCAL}/dataset/metadata.csv",
    "--data.audio_dir", f"{LOCAL}/dataset/wavs/",
    "--model.sample_rate", "22050",
    "--data.espeak_voice", ESPEAK_VOICE,
    "--data.cache_dir", f"{LOCAL}/cache/",
    "--data.config_path", CONFIG,
    "--data.batch_size", str(BATCH_SIZE),
    "--trainer.max_epochs", str(MAX_EPOCHS),
    "--trainer.max_time", SESSION_MAX_TIME,
    "--trainer.default_root_dir", LOGS,
] + start

print(" ".join(command))
process = subprocess.Popen(command, cwd="/content/piper1-gpl", stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
last_sync = time.time()

try:
    for line in process.stdout:
        print(line, end="")

        if time.time() - last_sync > SYNC_EVERY_S:
            sync()
            last_sync = time.time()
finally:
    process.wait()
    sync()

print("Código de salida:", process.returncode)

## 5. Probar y exportar a .onnx

In [ ]:
# Exporta el último checkpoint (o pon aquí otro: CKPT = "/content/drive/MyDrive/asia_voz/checkpoints/otro.ckpt").
CKPT = CKPT_DRIVE
ONNX = f"{DRIVE_DIR}/salida/{VOICE_NAME}-medium.onnx"
assert os.path.exists(CKPT), "No hay checkpoint: entrena primero"
assert os.path.exists(CONFIG), "Falta el JSON de la voz (lo escribe el entrenamiento)"

!python3 -m piper.train.export_onnx --checkpoint {CKPT} --output-file {ONNX}
print(os.listdir(f"{DRIVE_DIR}/salida"))

In [ ]:
# Una frase de prueba con la voz exportada.
from IPython.display import Audio, display

TEXT = "Hola, soy Asia. Esta es mi voz nueva; dime si se entiende bien."
!python3 -m piper -m {ONNX} -c {CONFIG} -f /content/prueba.wav -- "{TEXT}"
display(Audio("/content/prueba.wav"))

In [ ]:
# Baja los dos archivos a tu PC: van a data/voices/ de Asia (docs/VOZ_PROPIA.md, "Probarla en Asia").
from google.colab import files

files.download(ONNX)
files.download(CONFIG)

## 6. Borrar los audios

Borra el zip de `entrada/` en Drive y los audios de Colab. **No** borra los checkpoints ni la voz exportada
(`checkpoints/` y `salida/`): sin los audios no se puede seguir entrenando desde cero, pero sí reanudar con los
checkpoints si vuelves a subir el zip.

Drive manda lo borrado a la **Papelera**: vacíala en https://drive.google.com/drive/trash para que de verdad se vaya.

In [ ]:
DELETE_AUDIO = True   # pon False si todavía vas a reanudar el entrenamiento con este zip

if DELETE_AUDIO:
    shutil.rmtree(f"{DRIVE_DIR}/entrada", ignore_errors=True)
    shutil.rmtree(f"{LOCAL}/dataset", ignore_errors=True)
    shutil.rmtree(f"{LOCAL}/cache", ignore_errors=True)
    os.makedirs(f"{DRIVE_DIR}/entrada", exist_ok=True)
    left = [p for p in glob.glob(f"{DRIVE_DIR}/**/*.wav", recursive=True) + glob.glob(f"{DRIVE_DIR}/**/*.zip", recursive=True)]
    print("Audios que quedan en Drive:", left)
    print("Ahora vacía la Papelera de Drive: https://drive.google.com/drive/trash")
else:
    print("No se borró nada (DELETE_AUDIO = False).")